
# DeepONet Operator-Style Baseline for Fisher-Regularized Variance Dynamics

This notebook computes the comparison table for Direct MLP, mechanism-aware Residual MLP, Physics-informed Residual MLP, and DeepONet.

The reduced Fisher-regularized variance dynamics are

\[
\dot{u}=2(1-u)+\frac{\varepsilon}{u}.
\]

DeepONet is used as a lightweight operator-style baseline for the scalar map

\[
(u_0,\varepsilon,t)\mapsto u(t;u_0,\varepsilon),
\]

not as a full Fourier/operator-learning model over spatial density fields.


## Imports

In [ ]:

import math, random
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
try:
    from scipy.stats import t as student_t
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())


PyTorch: 2.11.0+cpu CUDA: False


## Configuration

In [ ]:

CONFIG = {
    'n_pairs': 300,
    'n_time': 80,
    't_max': 5.0,
    'u0_min': 0.2,
    'u0_max': 2.5,
    'eps_min': 0.01,
    'eps_max': 0.5,
    'n_seeds': 10,
    'data_seed': 123,
    'split_seed': 999,
    'base_seed': 2026,
    'epochs': 1500,
    'batch_size': 512,
    'lr': 2e-3,
    'patience': 150,
    # Use the best lambda from the physics-informed residual experiment.
    'physics_lambda': 0.1,
    'deeponet_p': 64,
    'quick_test': False,
}
if CONFIG['quick_test']:
    CONFIG.update({'n_pairs': 80, 'n_time': 50, 'n_seeds': 2, 'epochs': 200, 'patience': 40, 'deeponet_p': 32})
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE


'cpu'

## Dynamics and Dataset

In [ ]:

def u_star(eps):
    eps = np.asarray(eps)
    return (1.0 + np.sqrt(1.0 + 2.0 * eps)) / 2.0

def u_ou_np(t, u0):
    return 1.0 + (u0 - 1.0) * np.exp(-2.0 * t)

def u_ou_torch(t, u0):
    return 1.0 + (u0 - 1.0) * torch.exp(-2.0 * t)

def rhs(u, eps):
    return 2.0 * (1.0 - u) + eps / np.maximum(u, 1e-12)

def integrate_rk4(u0, eps, t_grid):
    u = np.zeros_like(t_grid, dtype=np.float64)
    u[0] = float(u0)
    for k in range(1, len(t_grid)):
        dt = t_grid[k] - t_grid[k-1]
        uk = u[k-1]
        k1 = rhs(uk, eps)
        k2 = rhs(uk + 0.5*dt*k1, eps)
        k3 = rhs(uk + 0.5*dt*k2, eps)
        k4 = rhs(uk + dt*k3, eps)
        u[k] = uk + dt*(k1 + 2*k2 + 2*k3 + k4)/6.0
        u[k] = max(u[k], 1e-12)
    return u

def make_dataset(n_pairs, n_time, u0_range=(0.2,2.5), eps_range=(0.01,0.5), t_max=5.0, seed=123):
    rng = np.random.default_rng(seed)
    u0s = rng.uniform(*u0_range, size=n_pairs)
    epss = rng.uniform(*eps_range, size=n_pairs)
    t_grid = np.linspace(0.0, t_max, n_time)
    rows = []
    for pair_id, (u0_i, eps_i) in enumerate(zip(u0s, epss)):
        u_clean = integrate_rk4(u0_i, eps_i, t_grid)
        for t_i, u_i in zip(t_grid, u_clean):
            rows.append({'pair_id': pair_id, 'u0': u0_i, 'eps': eps_i, 't': t_i, 'u_clean': u_i})
    return pd.DataFrame(rows)

def split_pair_ids(n_pairs, train_frac=0.70, val_frac=0.15, seed=999):
    rng = np.random.default_rng(seed)
    ids = np.arange(n_pairs)
    rng.shuffle(ids)
    n_train = int(train_frac*n_pairs)
    n_val = int(val_frac*n_pairs)
    return set(ids[:n_train]), set(ids[n_train:n_train+n_val]), set(ids[n_train+n_val:])

df = make_dataset(CONFIG['n_pairs'], CONFIG['n_time'], (CONFIG['u0_min'], CONFIG['u0_max']), (CONFIG['eps_min'], CONFIG['eps_max']), CONFIG['t_max'], CONFIG['data_seed'])
train_ids, val_ids, test_ids = split_pair_ids(CONFIG['n_pairs'], seed=CONFIG['split_seed'])
train_df = df[df.pair_id.isin(train_ids)].copy()
val_df = df[df.pair_id.isin(val_ids)].copy()
test_df = df[df.pair_id.isin(test_ids)].copy()
print(train_df.shape, val_df.shape, test_df.shape)
df.head()


(16800, 5) (3600, 5) (3600, 5)


,pair_id,u0,eps,t,u_clean
0,0,1.769409,0.145374,0.000000,1.769409
1,0,1.769409,0.145374,0.063291,1.682941
2,0,1.769409,0.145374,0.126582,1.607001
3,0,1.769409,0.145374,0.189873,1.540327
4,0,1.769409,0.145374,0.253165,1.481808


## Utilities and Models

In [ ]:

@dataclass
class Scaler:
    mean: np.ndarray
    std: np.ndarray
    def transform(self, x):
        return (x - self.mean) / self.std
    def inverse(self, x):
        return x * self.std + self.mean

def fit_scaler(x):
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    std = np.where(std < 1e-12, 1.0, std)
    return Scaler(mean, std)

def scaler_to_torch(scaler, device):
    return (torch.tensor(scaler.mean, dtype=torch.float32, device=device),
            torch.tensor(scaler.std, dtype=torch.float32, device=device))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

class MLP(nn.Module):
    def __init__(self, input_dim=3, hidden=(64,64), activation='tanh'):
        super().__init__()
        act = nn.ReLU if activation == 'relu' else nn.GELU if activation == 'gelu' else nn.Tanh
        layers=[]; prev=input_dim
        for h in hidden:
            layers += [nn.Linear(prev,h), act()]
            prev=h
        layers.append(nn.Linear(prev,1))
        self.net = nn.Sequential(*layers)
    def forward(self,x):
        return self.net(x)

class DeepONet(nn.Module):
    def __init__(self, branch_dim=2, trunk_dim=1, p=64, hidden=(64,64), activation='tanh'):
        super().__init__()
        act = nn.ReLU if activation == 'relu' else nn.GELU if activation == 'gelu' else nn.Tanh
        def make_net(input_dim, output_dim):
            layers=[]; prev=input_dim
            for h in hidden:
                layers += [nn.Linear(prev,h), act()]
                prev=h
            layers.append(nn.Linear(prev, output_dim))
            return nn.Sequential(*layers)
        self.branch = make_net(branch_dim, p)
        self.trunk = make_net(trunk_dim, p)
        self.bias = nn.Parameter(torch.zeros(1))
    def forward(self, branch_x, trunk_x):
        b = self.branch(branch_x)
        t = self.trunk(trunk_x)
        return torch.sum(b*t, dim=1, keepdim=True) + self.bias


## Direct and Residual MLP Training

In [ ]:

def train_scaled_mlp(x_train, y_train, x_val, y_val, seed, epochs=1500, batch_size=512, lr=2e-3, patience=150, device='cpu'):
    set_seed(seed)
    x_scaler = fit_scaler(x_train); y_scaler = fit_scaler(y_train)
    x_train_s = x_scaler.transform(x_train).astype(np.float32); y_train_s = y_scaler.transform(y_train).astype(np.float32)
    x_val_s = x_scaler.transform(x_val).astype(np.float32); y_val_s = y_scaler.transform(y_val).astype(np.float32)
    loader = DataLoader(TensorDataset(torch.tensor(x_train_s), torch.tensor(y_train_s)), batch_size=batch_size, shuffle=True)
    model = MLP(input_dim=x_train.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.MSELoss()
    xv = torch.tensor(x_val_s, dtype=torch.float32, device=device); yv = torch.tensor(y_val_s, dtype=torch.float32, device=device)
    best_state=None; best_val=float('inf'); wait=0
    for epoch in range(epochs):
        model.train()
        for xb,yb in loader:
            xb=xb.to(device); yb=yb.to(device)
            opt.zero_grad(); loss=loss_fn(model(xb), yb); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): val_loss = loss_fn(model(xv), yv).item()
        if val_loss < best_val:
            best_val = val_loss; best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait += 1
        if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return model, x_scaler, y_scaler

def predict_scaled_mlp(model, x, x_scaler, y_scaler, device='cpu'):
    model.eval(); xs = x_scaler.transform(x).astype(np.float32)
    with torch.no_grad(): pred_s = model(torch.tensor(xs, dtype=torch.float32, device=device)).cpu().numpy()
    return y_scaler.inverse(pred_s)


## Physics-Informed Residual MLP

In [ ]:

def train_physics_informed_residual(train_df, val_df, lam, seed, epochs=1500, batch_size=512, lr=2e-3, patience=150, device='cpu'):
    set_seed(seed)
    x_train = train_df[['u0','eps','t']].to_numpy(dtype=np.float64)
    x_val = val_df[['u0','eps','t']].to_numpy(dtype=np.float64)
    y_train_u = train_df[['u_clean']].to_numpy(dtype=np.float64)
    y_val_u = val_df[['u_clean']].to_numpy(dtype=np.float64)
    train_ou = u_ou_np(train_df.t.to_numpy(), train_df.u0.to_numpy())
    y_train_r = ((train_df.u_clean.to_numpy() - train_ou) / train_df.eps.to_numpy()).reshape(-1,1)
    x_scaler = fit_scaler(x_train); r_scaler = fit_scaler(y_train_r)
    x_mean, x_std = scaler_to_torch(x_scaler, device); r_mean, r_std = scaler_to_torch(r_scaler, device)
    loader = DataLoader(TensorDataset(torch.tensor(x_train.astype(np.float32)), torch.tensor(y_train_u.astype(np.float32))), batch_size=batch_size, shuffle=True)
    model = MLP(input_dim=3).to(device); opt = torch.optim.Adam(model.parameters(), lr=lr)
    xv = torch.tensor(x_val.astype(np.float32), device=device); yv = torch.tensor(y_val_u.astype(np.float32), device=device)
    best_state=None; best_val=float('inf'); wait=0
    for epoch in range(epochs):
        model.train()
        for xb_raw_cpu, yb_cpu in loader:
            xb_raw = xb_raw_cpu.to(device).clone().detach().requires_grad_(True)
            yb = yb_cpu.to(device)
            u0 = xb_raw[:,0:1]; eps = xb_raw[:,1:2]; t = xb_raw[:,2:3]
            xb_norm = (xb_raw - x_mean)/x_std
            r_scaled = model(xb_norm); r_pred = r_scaled*r_std + r_mean
            u_hat = u_ou_torch(t,u0) + eps*r_pred
            u_safe = torch.clamp(u_hat, min=1e-6)
            loss_traj = torch.mean((u_hat-yb)**2)
            du_dt = torch.autograd.grad(outputs=u_hat.sum(), inputs=xb_raw, create_graph=True, retain_graph=True)[0][:,2:3]
            ode_rhs = 2.0*(1.0-u_safe) + eps/u_safe
            loss_phys = torch.mean((du_dt-ode_rhs)**2)
            loss = loss_traj + lam*loss_phys
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            xvn=(xv-x_mean)/x_std; u0v=xv[:,0:1]; epsv=xv[:,1:2]; tv=xv[:,2:3]
            rv = model(xvn)*r_std + r_mean
            uh = u_ou_torch(tv,u0v) + epsv*rv
            val_loss = torch.mean((uh-yv)**2).item()
        if val_loss < best_val:
            best_val = val_loss; best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait += 1
        if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return model, x_scaler, r_scaler

def predict_physics_informed_residual(model, x, x_scaler, r_scaler, device='cpu'):
    model.eval(); xr = torch.tensor(x.astype(np.float32), device=device)
    x_mean,x_std = scaler_to_torch(x_scaler,device); r_mean,r_std = scaler_to_torch(r_scaler,device)
    with torch.no_grad():
        xn=(xr-x_mean)/x_std; u0=xr[:,0:1]; eps=xr[:,1:2]; t=xr[:,2:3]
        rp = model(xn)*r_std + r_mean
        uh = u_ou_torch(t,u0) + eps*rp
    return uh.cpu().numpy()


## DeepONet Training

In [ ]:

def train_deeponet(train_df, val_df, seed, p=64, epochs=1500, batch_size=512, lr=2e-3, patience=150, device='cpu'):
    set_seed(seed)
    b_train = train_df[['u0','eps']].to_numpy(dtype=np.float64); t_train = train_df[['t']].to_numpy(dtype=np.float64); y_train = train_df[['u_clean']].to_numpy(dtype=np.float64)
    b_val = val_df[['u0','eps']].to_numpy(dtype=np.float64); t_val = val_df[['t']].to_numpy(dtype=np.float64); y_val = val_df[['u_clean']].to_numpy(dtype=np.float64)
    b_scaler = fit_scaler(b_train); t_scaler = fit_scaler(t_train); y_scaler = fit_scaler(y_train)
    bs = b_scaler.transform(b_train).astype(np.float32); ts = t_scaler.transform(t_train).astype(np.float32); ys = y_scaler.transform(y_train).astype(np.float32)
    bvs = b_scaler.transform(b_val).astype(np.float32); tvs = t_scaler.transform(t_val).astype(np.float32); yvs = y_scaler.transform(y_val).astype(np.float32)
    loader = DataLoader(TensorDataset(torch.tensor(bs), torch.tensor(ts), torch.tensor(ys)), batch_size=batch_size, shuffle=True)
    model = DeepONet(p=p).to(device); opt=torch.optim.Adam(model.parameters(), lr=lr); loss_fn=nn.MSELoss()
    bv=torch.tensor(bvs,dtype=torch.float32,device=device); tv=torch.tensor(tvs,dtype=torch.float32,device=device); yv=torch.tensor(yvs,dtype=torch.float32,device=device)
    best_state=None; best_val=float('inf'); wait=0
    for epoch in range(epochs):
        model.train()
        for bb,tt,yy in loader:
            bb=bb.to(device); tt=tt.to(device); yy=yy.to(device)
            opt.zero_grad(); loss=loss_fn(model(bb,tt), yy); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): val_loss=loss_fn(model(bv,tv), yv).item()
        if val_loss < best_val:
            best_val=val_loss; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait += 1
        if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return model, b_scaler, t_scaler, y_scaler

def predict_deeponet(model, df_input, b_scaler, t_scaler, y_scaler, device='cpu'):
    model.eval()
    b = df_input[['u0','eps']].to_numpy(dtype=np.float64); t = df_input[['t']].to_numpy(dtype=np.float64)
    bs=b_scaler.transform(b).astype(np.float32); ts=t_scaler.transform(t).astype(np.float32)
    with torch.no_grad(): pred_s = model(torch.tensor(bs,dtype=torch.float32,device=device), torch.tensor(ts,dtype=torch.float32,device=device)).cpu().numpy()
    return y_scaler.inverse(pred_s)


## Evaluation and Run

In [ ]:

def crossing_time(t,u,level=1.0):
    if u[0] >= level: return None
    above = np.where(u >= level)[0]
    if len(above) == 0: return None
    k=above[0]
    if k==0: return t[0]
    t0,t1=t[k-1],t[k]; u0,u1=u[k-1],u[k]
    if abs(u1-u0)<1e-12: return t1
    return t0 + (level-u0)*(t1-t0)/(u1-u0)

def compute_metrics(test_pred_df):
    e_traj=[]; e_cross=[]; e_over=[]; e_eq=[]
    for pair_id,g in test_pred_df.groupby('pair_id'):
        g=g.sort_values('t')
        t=g.t.to_numpy(); ut=g.u_clean.to_numpy(); up=g.u_pred.to_numpy(); eps=float(g.eps.iloc[0]); ueq=float(u_star(np.array([eps]))[0])
        e_traj.append(np.mean((up-ut)**2))
        tc_true = crossing_time(t,ut)
        if tc_true is not None:
            tc_pred = crossing_time(t,up)
            if tc_pred is None: tc_pred=t[-1]
            e_cross.append(abs(tc_pred-tc_true))
        e_over.append(abs(max(0.0,float(np.max(up)-ueq)) - max(0.0,float(np.max(ut)-ueq))))
        e_eq.append(abs(float(up[-1])-ueq))
    return {'E_traj':float(np.mean(e_traj)), 'E_cross':float(np.mean(e_cross)) if e_cross else np.nan, 'E_over':float(np.mean(e_over)), 'E_eq':float(np.mean(e_eq))}

def run_deeponet_baseline_comparison(config=CONFIG, device=DEVICE):
    rows=[]
    x_train=train_df[['u0','eps','t']].to_numpy(dtype=np.float64); x_val=val_df[['u0','eps','t']].to_numpy(dtype=np.float64); x_test=test_df[['u0','eps','t']].to_numpy(dtype=np.float64)
    y_train=train_df[['u_clean']].to_numpy(dtype=np.float64); y_val=val_df[['u_clean']].to_numpy(dtype=np.float64)
    train_ou=u_ou_np(train_df.t.to_numpy(), train_df.u0.to_numpy()); val_ou=u_ou_np(val_df.t.to_numpy(), val_df.u0.to_numpy()); test_ou=u_ou_np(test_df.t.to_numpy(), test_df.u0.to_numpy())
    y_train_res=((train_df.u_clean.to_numpy()-train_ou)/train_df.eps.to_numpy()).reshape(-1,1)
    y_val_res=((val_df.u_clean.to_numpy()-val_ou)/val_df.eps.to_numpy()).reshape(-1,1)
    for seed_idx in range(config['n_seeds']):
        run_seed=config['base_seed']+seed_idx
        print(f'\nSeed {seed_idx:02d}')
        dm,dxs,dys=train_scaled_mlp(x_train,y_train,x_val,y_val,run_seed,config['epochs'],config['batch_size'],config['lr'],config['patience'],device)
        pred=predict_scaled_mlp(dm,x_test,dxs,dys,device).reshape(-1); pdf=test_df.copy(); pdf['u_pred']=pred; m=compute_metrics(pdf); rows.append({'model':'Direct MLP','seed':seed_idx,**m})
        rm,rxs,rys=train_scaled_mlp(x_train,y_train_res,x_val,y_val_res,run_seed+5000,config['epochs'],config['batch_size'],config['lr'],config['patience'],device)
        rpred=predict_scaled_mlp(rm,x_test,rxs,rys,device).reshape(-1); pred=test_ou+test_df.eps.to_numpy()*rpred; pdf=test_df.copy(); pdf['u_pred']=pred; m=compute_metrics(pdf); rows.append({'model':'Residual MLP','seed':seed_idx,**m})
        lam=config['physics_lambda']
        pim,pixs,pirs=train_physics_informed_residual(train_df,val_df,lam,run_seed+int(lam*10000)+9000,config['epochs'],config['batch_size'],config['lr'],config['patience'],device)
        pred=predict_physics_informed_residual(pim,x_test,pixs,pirs,device).reshape(-1); pdf=test_df.copy(); pdf['u_pred']=pred; m=compute_metrics(pdf); rows.append({'model':'Physics-informed Residual MLP','seed':seed_idx,**m})
        dom,bs,ts,ys=train_deeponet(train_df,val_df,run_seed+12000,config['deeponet_p'],config['epochs'],config['batch_size'],config['lr'],config['patience'],device)
        pred=predict_deeponet(dom,test_df,bs,ts,ys,device).reshape(-1); pdf=test_df.copy(); pdf['u_pred']=pred; m=compute_metrics(pdf); rows.append({'model':'DeepONet','seed':seed_idx,**m})
        print('  done')
    return pd.DataFrame(rows)

deeponet_seed_results = run_deeponet_baseline_comparison()
deeponet_seed_results.head()



Seed 00
  done

Seed 01
  done

Seed 02
  done

Seed 03
  done

Seed 04
  done

Seed 05
  done

Seed 06
  done

Seed 07
  done

Seed 08
  done

Seed 09
  done


,model,seed,E_traj,E_cross,E_over,E_eq
0,Direct MLP,0,1.044728e-07,0.001061,0.000348,0.000300
1,Residual MLP,0,4.051397e-09,0.000218,0.000064,0.000029
2,Physics-informed Residual MLP,0,1.333785e-08,0.000381,0.000082,0.000085
3,DeepONet,0,5.933270e-07,0.004749,0.000605,0.000306
4,Direct MLP,1,1.042909e-07,0.001574,0.000200,0.000298


## Summary and LaTeX Table

In [ ]:

def summarize_results(seed_df, n_seeds):
    tcrit = float(student_t.ppf(0.975, n_seeds-1)) if SCIPY_AVAILABLE else 2.262
    model_order=['Direct MLP','Residual MLP','Physics-informed Residual MLP','DeepONet']
    rows=[]
    for model in model_order:
        g=seed_df[seed_df.model==model]
        row={'model':model}
        for metric in ['E_traj','E_cross','E_over','E_eq']:
            vals=g[metric].to_numpy(dtype=float); mean=np.nanmean(vals); std=np.nanstd(vals,ddof=1); n=np.sum(~np.isnan(vals)); ci=tcrit*std/math.sqrt(n) if n>1 else np.nan
            row[f'{metric}_mean']=mean; row[f'{metric}_ci']=ci
        rows.append(row)
    return pd.DataFrame(rows)

def fmt_metric(mean, ci):
    if np.isnan(mean): return 'N/A'
    if np.isnan(ci): return f'${mean:.3e}$'
    return f'${mean:.3e} \\pm {ci:.1e}$'

def make_deeponet_latex_table(summary_df):
    lines=[]
    lines.append(r'\begin{table}[t]')
    lines.append(r'\centering')
    lines.append(r'\scriptsize')
    lines.append(r'\caption{Comparison with a lightweight DeepONet operator-style baseline. DeepONet uses a branch network for $(u_{0},\varepsilon)$ and a trunk network for $t$ to predict $u(t;u_{0},\varepsilon)$. Results are reported as mean $\pm$ 95\% confidence-interval half-width over $S=10$ random seeds. Lower values indicate better performance.}')
    lines.append(r'\label{tab:deeponet_baseline}')
    lines.append(r'\begin{tabular}{lcccc}')
    lines.append(r'\hline')
    lines.append(r'Model & $E_{\mathrm{traj}}$ & $E_{\mathrm{cross}}$ & $E_{\mathrm{over}}$ & $E_{\mathrm{eq}}$ \\')
    lines.append(r'\hline')
    for _,row in summary_df.iterrows():
        vals=[fmt_metric(row['E_traj_mean'],row['E_traj_ci']), fmt_metric(row['E_cross_mean'],row['E_cross_ci']), fmt_metric(row['E_over_mean'],row['E_over_ci']), fmt_metric(row['E_eq_mean'],row['E_eq_ci'])]
        lines.append(f"{row['model']} & {vals[0]} & {vals[1]} & {vals[2]} & {vals[3]} \\\")
    lines.append(r'\hline')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)

deeponet_summary = summarize_results(deeponet_seed_results, CONFIG['n_seeds'])
deeponet_latex_table = make_deeponet_latex_table(deeponet_summary)
print(deeponet_latex_table)


SyntaxError: unterminated f-string literal (detected at line 32) (1288641687.py, line 32)

## Save Results

In [ ]:

deeponet_seed_results.to_csv('deeponet_baseline_seed_results.csv', index=False)
deeponet_summary.to_csv('deeponet_baseline_summary.csv', index=False)
with open('deeponet_baseline_latex_table.tex', 'w', encoding='utf-8') as f:
    f.write(deeponet_latex_table)
print('Saved:')
print('  deeponet_baseline_seed_results.csv')
print('  deeponet_baseline_summary.csv')
print('  deeponet_baseline_latex_table.tex')
